# HS4002 Week 6

## OLS with Higher-Order Effects, Interaction Terms, and Logistic Regression

Today we cover:
1. Polynomial terms in OLS (age-squared)
2. Interaction effects
3. The Linear Probability Model (LPM)
4. Logistic regression
5. Marginal effects


In [2]:
# !pip install pandas numpy plotnine statsmodels marginaleffects stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [3]:
%pip install stargazer
from stargazer.stargazer import Stargazer

Note: you may need to restart the kernel to use updated packages.


In [4]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

# OLS with Higher-Order Polynomials

Begin by regressing income on age. Save as `ols_model1`.

In [5]:
ols_model1 = smf.ols('income_cont ~ age', data=df).fit()
ols_model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                   0.01847
Date:                Sun, 20 Sep 2026   Prob (F-statistic):              0.892
Time:                        18:52:38   Log-Likelihood:                -8910.5
No. Observations:                 707   AIC:                         1.782e+04
Df Residuals:                     705   BIC:                         1.783e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8.632e+04   8546.726     10.099      0.000    6.95e+04    1.03e+05
age           21.2431    156.330      0.136      0.892    -285.684     328.171
==============================================================================
Omnibus:                      113.766   Durbin-Watson:                   1.629
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              168.872
Skew:                           1.178   Prob(JB):                     2.14e-37
Kurtosis:                       3.430   Cond. No.                         172.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Adding an age-squared term

In Python formulas, use `I(age**2)` — the `I()` wrapper tells the formula parser to treat the expression literally.

In [6]:
ols_model2 = smf.ols('income_cont ~ age + I(age**2)', data=df).fit()
ols_model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     7.568
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           0.000560
Time:                        18:52:44   Log-Likelihood:                -8903.0
No. Observations:                 707   AIC:                         1.781e+04
Df Residuals:                     704   BIC:                         1.783e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -2088.6030   2.43e+04     -0.086      0.931   -4.97e+04    4.55e+04
age          3886.3291   1006.049      3.863      0.000    1911.113    5861.545
I(age ** 2)   -37.4677      9.636     -3.888      0.000     -56.387     -18.548
==============================================================================
Omnibus:                      109.972   Durbin-Watson:                   1.611
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              160.918
Skew:                           1.148   Prob(JB):                     1.14e-35
Kurtosis:                       3.440   Cond. No.                     3.16e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.16e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Making a regression table

As in Week 5, the `stargazer` package collects fitted models into a single regression table. Reading models side by side is much easier than scrolling through two separate `.summary()` blocks.

In [7]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_model1, ols_model2])
sg.title('Income, Age, and Age-Squared')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 1', 'Model 2'], [1, 1])
sg.covariate_order(['age', 'I(age ** 2)', 'Intercept'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
	'Intercept':   'Intercept'
})

# Ending the cell on the object itself renders the table in the notebook
sg

In [ ]:
# This is a Linear regression model that predicts income from age and age-squared.
    # The dependent variable is income, it is a continuous variable, measured in USD. One unit change correspond to one dollars change in income.
    # The independent variable is age and age-squared. Age is a continuous variable, measured in years. One unit change correspond to one year change in age.
    # Age-squared is the quadratic term of age.
# According to model1
    # There is no linear association between age and income (p>0.1), ceteris paribus.
# According to model2
    # After adding the squared term of age, suddenly age become significant. As the coefficient of age is positive, while the coefficient of age-squared is negative, this implies a concave relationship between age and income.
    # The association between age and income depends on the person's current age. Predicted income initially increases with age, but the increase becomes smaller. After reach the turning point, the predicted income decreases as age increases.
    # The p-value tell us: if the true value of age-squared coefficient is 0, the probability we obtain coefficient of age-squared or more extreme value is less than 0.01.
    # The p-value is small, this is quite unlikely to happen under null, thus we reject null and infer there is a concave relationship between age and income.

In [8]:
person_a = 15
person_b = 195
(person_b - person_a)*3886 + ((person_b - person_a)**2)*(-37.5) #According to this model, what is the expected difference between person A and person B
# The main effect is positive, the square term is negative

-515520.0

## Interaction Effects

Add BA education and gender to the model. Save as `ols_model3`.

In [9]:
ols_model3 = smf.ols('income_cont ~ age + I(age**2) + ba_binary + sex_woman', data=df).fit()
ols_model3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     44.79
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           1.61e-33
Time:                        18:53:01   Log-Likelihood:                -8830.1
No. Observations:                 707   AIC:                         1.767e+04
Df Residuals:                     702   BIC:                         1.769e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept          -8893.2601   2.21e+04     -0.402      0.688   -5.23e+04    3.46e+04
sex_woman[T.Woman] -1.663e+04   4871.657     -3.414      0.001   -2.62e+04   -7067.441
age                 3275.5520    910.156      3.599      0.000    1488.599    5062.505
I(age ** 2)          -31.3456      8.720     -3.595      0.000     -48.466     -14.225
ba_binary            6.04e+04   4874.146     12.393      0.000    5.08e+04       7e+04
==============================================================================
Omnibus:                      102.337   Durbin-Watson:                   1.765
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              145.602
Skew:                           1.046   Prob(JB):                     2.42e-32
Kurtosis:                       3.751   Cond. No.                     3.19e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.19e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Now add an interaction term between BA and gender. In statsmodels formulas, `:` creates an interaction.

Save as `ols_model4`.

In [10]:
import statsmodels.formula.api as smf

ols_model4 = smf.ols(
	'income_cont ~ age + I(age**2) + ba_binary + sex_woman + ba_binary:sex_woman',
	data=df
).fit()

In [11]:
sg = Stargazer([ols_model3, ols_model4])
sg.title('Interaction Between BA and Gender')
sg.dependent_variable_name('Income (USD)')

sg.custom_columns(['Model 3', 'Model 4'], [1, 1])
sg.covariate_order([
	'age',
	'I(age ** 2)',
	'ba_binary',
	'sex_woman[T.Woman]',
	'ba_binary:sex_woman[T.Woman]',
	'Intercept'
])
sg.rename_covariates({
	'age':                          'Age',
	'I(age ** 2)':                  'Age-squared',
	'ba_binary':                    'BA degree',
	'sex_woman[T.Woman]':           'Gender (woman = 1)',
	'ba_binary:sex_woman[T.Woman]': 'BA × Woman',
	'Intercept':                    'Intercept'
})

sg

In [ ]:
# Model3: BA degree is coded as binary variable, there is expected difference between someone with BA degree and someone who are not
# Model3: The expected difference between women and not women is 16632.204 dollars
# Model4: implication: the main difference between men and women is actually between payoff of education between women and men
# --> if you are a women with a BA degree, you will make 8323.376 + 17484.329 dollars less
# Interaction effect build on top of main effect

In [ ]:
# Reference group: men, people without bachelor degree.
# Apart from main effect of bachelor degree and gender, we also observed an interaction effect between BA degree and gender.
# For women, having a bachelor degree on average is associated with a $17484 dollar decreace in income, compared to the reference group (men) (p<0.1), ceteris paribus.
# The p-value tell us that: if the true value of this interact coefficient is 0, the probability we obtain a interaction coefficient = -17484 or more extreme value is less than 0.1.
# The p-value is small, this is quite unlikely to happen under null, thus we reject H0 and infer the association between having a bachlor degree and income differs by gender.

# Using the Linear Probability Model (LPM)

The LPM uses OLS on a binary dependent variable. It's quick and interpretable, though it can predict probabilities outside [0, 1].

Regress live music attendance (`binary_lvmus`) on BA education and log income.

In [12]:
lpm_model1 = smf.ols(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(lpm_model1.summary())

                            OLS Regression Results                            
Dep. Variable:           binary_lvmus   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.066
Method:                 Least Squares   F-statistic:                     26.08
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           1.19e-11
Time:                        19:25:41   Log-Likelihood:                -486.80
No. Observations:                 707   AIC:                             979.6
Df Residuals:                     704   BIC:                             993.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.5077    

In [ ]:
# We observe that on average being people with a BA degree is associated with a expected 0.1280 increase in the probability of live music attendence, compared to people without a BA degree (p=0.001), ceteris paribus.
    # We observe that on average 0.1280 is the expected difference in the probability of live music attendence between people with BA degree and people without BA degree (p=0.001), ceteris paribus. 
# We observe that on average 0.0887 is the expected change in the probability of live music attendence when the log income increases by 1 unit (p<0.01), ceteris paribus.
# The p-value tell us if the true value of the coefficient of log income is 0, the probability we obtain a coefficient of log income = 0.0887 or more extreme value is less than 0.001.
# The p-value is small, it is unlikely to happen under null, thus we reject H0 and infer log coefficient of income is not zero with sufficient evidence.

# Using a Logit Model

Now, let's try logistic regression.

In [17]:
logit_model1 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont)',
	data=df
).fit()
print(logit_model1.summary())

Optimization terminated successfully.
         Current function value: 0.655998
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      704
Method:                           MLE   Df Model:                            2
Date:                Wed, 16 Sep 2026   Pseudo R-squ.:                 0.05151
Time:                        11:56:58   Log-Likelihood:                -463.79
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 1.151e-11
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -4.4595      0.961     -4.639      0.000      -6.344      -2.575
ba

In [ ]:
# Why difference in coefficient? Capture different thing.
    # The coefficients for LPM and Logit model are different because they measure effects on different scales.
# We observe that on average being people with BA degree is expected to cause an increase in log odds of live music attendence by 0.5186 (p=0.002), ceteris paribus.
# We observe that on average on unit increase in log income is expected to cuase an increase in log odds of live music attendence by 0.3938 (p<0.001), ceteris paribus.

Build a second logit model adding gender, age, and age-squared. Compare both.

In [18]:
logit_model2 = smf.logit(
	'binary_lvmus ~ ba_binary + np.log(income_cont) + sex_woman + age + I(age**2)',
	data=df
).fit()
print(logit_model2.summary())

Optimization terminated successfully.
         Current function value: 0.645323
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           binary_lvmus   No. Observations:                  707
Model:                          Logit   Df Residuals:                      701
Method:                           MLE   Df Model:                            5
Date:                Wed, 16 Sep 2026   Pseudo R-squ.:                 0.06695
Time:                        11:58:45   Log-Likelihood:                -456.24
converged:                       True   LL-Null:                       -488.98
Covariance Type:            nonrobust   LLR p-value:                 8.952e-13
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -3.6710      1.156     -3.176      0.001      -5.936      -1.405
se

In [19]:
# compare lpm and logit models using stargazer
sg = Stargazer([lpm_model1, logit_model1, logit_model2])
sg.title('Live Music Attendance: LPM vs Logit')
sg.dependent_variable_name('Attends live music')

sg.custom_columns(['LPM', 'Logit 1', 'Logit 2'], [1, 1, 1])
sg.covariate_order([
	'ba_binary',
	'np.log(income_cont)',
	'sex_woman[T.Woman]',
	'age',
	'I(age ** 2)',
	'Intercept'
])
sg.rename_covariates({
	'ba_binary':           'BA degree',
	'np.log(income_cont)': 'log(Income)',
	'sex_woman[T.Woman]':  'Gender (woman = 1)',
	'age':                 'Age',
	'I(age ** 2)':         'Age-squared',
	'Intercept':           'Intercept'
})

# The LPM coefficients are changes in probability, the logit coefficients are
# log-odds, so compare signs and significance across columns — not magnitudes.
sg



In [ ]:
# Model3 added in more independent variables, which is gender, age and age-squared. 
# We observed that on average, being women is associated with an increase in log odds of live music attendence by 0.438 (p<0.01), ceteris paribus.
# We observed that on average, on year increase in age is expected to cause a decrease in log odds of live music attendence by 0.058 (p<0.1), ceteris paribus.
# We observed that there is no quedratic effect between age and log odds of live music attendence.

## Comparing Logit Models

### AIC comparison

In [20]:
print(f'AIC logit_model1: {logit_model1.aic:.2f}')
print(f'AIC logit_model2: {logit_model2.aic:.2f}')

AIC logit_model1: 933.58
AIC logit_model2: 924.49


In [ ]:
# AIC comprison give penalties for having more variables and a smaller AIC value indicates a better model.
# As the logit_model2 has a smaller AIC value than logit_model1, this implies that model2 provides better fit, according to AIC.
# Therefore, the additional variables, gender, age and age-squared improves model's fit according to AIC.

### Likelihood ratio test


In [21]:
from scipy.stats import chi2

lr_stat = 2 * (logit_model2.llf - logit_model1.llf)
df_diff = logit_model2.df_model - logit_model1.df_model
p_value = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}')
print(f'df = {df_diff:.0f}')
print(f'p-value = {p_value:.4f}')

LR statistic = 15.0945
df = 3
p-value = 0.0017


In [ ]:
# Likelihood ratio test is testing the additional variables are necessary or not
# We estimate a F-statistic= 15.09, with a p-value = 0.0017
# H0: The true value of all additional variables' coefficient are 0; H1: The true value of all additional variables' coefficient are not 0.
# The p-value tell us: if the true value of all additional variables coefficient are 0 (H0), the probability that we obtain a F-statistic=15.09 or more extreme value is 0.0017.
# As p-value is small, this is unlikely to happen under null, thus we reject H0.

# Marginal Effects

The Python `marginaleffects` package mirrors R's package of the same name.

In [24]:
# !pip install marginaleffects
from marginaleffects import avg_comparisons, comparisons

In [23]:
%pip install marginaleffects
from marginaleffects import avg_comparisons, comparisons

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 MB 3.1 MB/s  0:00:20m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 5.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 2.3 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 2.7 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 3.0 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [marginaleffects] [jax]ib]c]ntime-32]
Note: you may need to restart the kernel to use updated packages.


## Average marginal effect

The average marginal effect of a BA degree — equivalent to R's `avg_comparisons(logit_model1, variables='ba_binary')`.

In [25]:
avg_comparisons(logit_model1, variables='ba_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.123828,0.040752,3.03855,0.002465,8.664095,0.043817,0.203839


Interpretation: on average across our sample, having a BA degree increases the **probability** of attending a live music concert by X percentage points, holding other variables constant.

In [ ]:
# We observed that on average across our sample, having a BA degree is associated with a increase the probability of attending a live music concert by 12.38 percentage points (p<0.01), ceteris paribus. 

## Marginal effect at the mean

The marginal effect for a hypothetical person at the average of all X variables.

In [26]:
comparisons(logit_model1, variables='ba_binary', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",0.126466,0.041026,3.082541,0.002133,8.873215,0.045917,0.207014


In [ ]:
# Interpretation
# We observe that for a hypothetical indivdual with all other covariates at their sample means, having a bachelor degree is associated with an increase in probability of attending a live music concert by 0.13 percentage points (p<0.01), ceteris paribus.

## Class Exercise 
Own example

In [27]:
df.columns

Index(['id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat', 'income16',
       'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'income_cont',
       'binary_lvmus', 'binary_artxbt', 'binary_movie', 'binary_creat', 'omni',
       'race_bin', 'ba_binary', 'sex_woman', 'working'],
      dtype='str')

In [29]:
raw_df = pd.read_csv("gss2022_mini.csv")

chosen_variables = [
	'id', 'age', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat'
]
df = raw_df[chosen_variables].dropna().copy()

# Recode income16 to continuous midpoint values
income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont'] = df['income16'].map(income_map)

# Binary cultural participation indicators
df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

# Derived variables
df['race_bin']   = df['race'].astype(str)
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = np.where(df['sex'] == 2, 'Woman', 'Not Woman')
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')

In [37]:
ols_modelone = smf.ols('income_cont ~ race_bin + ba_binary', data=df).fit()
ols_modelone.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.184
Model:                            OLS   Adj. R-squared:                  0.180
Method:                 Least Squares   F-statistic:                     52.72
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           9.60e-31
Time:                        14:34:46   Log-Likelihood:                -8838.8
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     703   BIC:                         1.770e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         6.11e+04   3666.396     16.665      0.000    5.39e+04    6.83e+04
race_bin[T.2.0] -1.949e+04   7706.330     -2.529      0.012   -3.46e+04   -4359.080
race_bin[T.3.0]  6659.9291   9707.695      0.686      0.493   -1.24e+04    2.57e+04
ba_binary        5.887e+04   4942.113     11.911      0.000    4.92e+04    6.86e+04
==============================================================================
Omnibus:                      105.480   Durbin-Watson:                   1.779
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              151.350
Skew:                           1.077   Prob(JB):                     1.36e-33
Kurtosis:                       3.703   Cond. No.                         4.57
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [38]:
ols_modeltwo = smf.ols(
	'income_cont ~ race_bin + ba_binary + ba_binary:race',
	data=df
).fit()
ols_modeltwo.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            income_cont   R-squared:                       0.184
Model:                            OLS   Adj. R-squared:                  0.179
Method:                 Least Squares   F-statistic:                     39.61
Date:                Wed, 16 Sep 2026   Prob (F-statistic):           6.23e-30
Time:                        14:35:12   Log-Likelihood:                -8838.6
No. Observations:                 707   AIC:                         1.769e+04
Df Residuals:                     702   BIC:                         1.771e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        6.177e+04   3817.066     16.184      0.000    5.43e+04    6.93e+04
race_bin[T.2.0] -2.152e+04   8341.221     -2.579      0.010   -3.79e+04   -5139.326
race_bin[T.3.0]  1049.4366   1.31e+04      0.080      0.936   -2.47e+04    2.68e+04
ba_binary          5.2e+04   1.19e+04      4.382      0.000    2.87e+04    7.53e+04
ba_binary:race   5515.2597   8664.338      0.637      0.525   -1.15e+04    2.25e+04
==============================================================================
Omnibus:                      105.169   Durbin-Watson:                   1.781
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              150.745
Skew:                           1.075   Prob(JB):                     1.85e-33
Kurtosis:                       3.702   Cond. No.                         10.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
print(ols_modelone.params.index)
print(ols_modeltwo.params.index)

Index(['Intercept', 'race_bin[T.2.0]', 'race_bin[T.3.0]', 'ba_binary'], dtype='str')
Index(['Intercept', 'race_bin[T.2.0]', 'race_bin[T.3.0]', 'ba_binary',
       'ba_binary:race'],
      dtype='str')


In [ ]:
print(df['race_bin'].value_counts())
print(df['race_bin'].unique())
# According to code book, white = 1, black = 2, other = 3.

race_bin
1.0    575
2.0     83
3.0     49
Name: count, dtype: int64
<StringArray>
['1.0', '2.0', '3.0']
Length: 3, dtype: str


In [46]:
import pandas as pd
pd.crosstab(df['race'], df['race_bin'], margins=True)

race_bin,1.0,2.0,3.0,All
race,,,,
1.0,575,0,0,575
2.0,0,83,0,83
3.0,0,0,49,49
All,575,83,49,707


In [ ]:
sg = Stargazer([ols_modelone, ols_modeltwo])
sg.title('Interaction Between Race and BA degree')
sg.dependent_variable_name('Income (USD)')
sg.custom_columns(['Model 1', 'Model 2'], [1, 1])

sg.covariate_order([
    'race_bin[T.2.0]',
    'race_bin[T.3.0]',
    'ba_binary',
    'ba_binary:race',
    'Intercept'
])

sg.rename_covariates({
    'race_bin[T.2.0]': 'Race (Group 2)',
    'race_bin[T.3.0]': 'Race (Group 3)',
    'ba_binary': 'BA degree',
    'ba_binary:race': 'BA × Race',
    'Intercept': 'Intercept'
})
sg

In [ ]:
# The interpretation:
    # Dependent variable is income(USD), which able to predict through race and bachelor degree
    # Independent variable is Race and bachelor degree
# Model1:
    # Race (Group2): we observe that on average being Black is associated with a -19489 dollars change in income when compared to the reference group (White) (p < 0.05), ceteris paribus
    # Race (Group3): we observe no associatin between other race and income (p > 0.1), ceteris paribus
    # Bachelor degree: we observe an expected difference between people with BA degree and without, people with a BA degree earns 58867 more than people without (p < 0.05), ceteris paribus
# Model2 (add an interaction between race and BA degree):
    # BA x Race: we observe no interaction effect between race and BA degree (p > 0.1), ceteris paribus